# Project: Hand-made filters and HOG 🖌️
Before CNNs *learned* their filters, people designed them by hand. Implement convolution yourself, apply classic kernels (blur, sharpen, Sobel edges) to a photo, then build **HOG** (histogram of oriented gradients), the feature behind 2005's pedestrian detectors, and see how it compares with raw pixels on digits.

Topic: [[Computer Vision Overview]], [[CNN]] · guide note: [[Project - Hand-Made Filters and HOG]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy.signal import correlate2d
from sklearn.datasets import load_sample_image, load_digits
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
img = load_sample_image("china.jpg").mean(2)[::3, ::3] / 255          # grey, 143×214

## 1. Convolution (well, cross-correlation, like every DL library)
'Valid' mode: slide the kernel over every position where it fits completely and sum the element-wise product. Output size `(H − kh + 1, W − kw + 1)`.

In [ ]:
def conv2d(img, kernel):
    # TODO 1: valid cross-correlation with two loops
    raise NotImplementedError  # TODO 1

In [ ]:
check("conv2d", lambda: np.allclose(conv2d(img[:30, :40], np.arange(9.0).reshape(3, 3)), correlate2d(img[:30, :40], np.arange(9.0).reshape(3, 3), mode="valid")),
      "Must match scipy.signal.correlate2d(..., mode='valid').")
KERNELS = {"blur": np.ones((5, 5)) / 25, "sharpen": np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0.0]]),
           "Sobel x (vertical edges)": np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1.0]]), "Sobel y (horizontal edges)": np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1.0]])}
if ready("conv2d"):
    fig, ax = plt.subplots(1, 5, figsize=(17, 3)); ax[0].imshow(img, cmap="gray"); ax[0].set_title("original")
    for a, (name, k) in zip(ax[1:], KERNELS.items()):
        out = conv2d(img, k); a.imshow(np.abs(out) if "Sobel" in name else out, cmap="gray"); a.set_title(name)
    for a in ax: a.axis("off")
    plt.show()

## 2. HOG
Compute gradients with central differences (`gx[:, 1:-1] = img[:, 2:] − img[:, :-2]`, same for `gy` along rows), magnitude and **unsigned** orientation in [0°, 180°). Split the image into `cell × cell` blocks; in each, build a `bins`-bin orientation histogram weighted by magnitude, L2-normalise it, and concatenate all cells.

In [ ]:
def hog(img, cell=4, bins=9):
    # TODO 2: gradients → per-cell magnitude-weighted orientation histograms → normalise → concatenate
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_hog():
    vert = np.zeros((8, 8)); vert[:, 4:] = 1                   # a vertical edge → horizontal gradient → angle 0°
    f = hog(vert, cell=4, bins=9)
    return f.shape == (36,) and f.reshape(4, 9)[1].argmax() == 0 and np.isclose(np.linalg.norm(f.reshape(4, 9)[1]), 1, atol=1e-4)
check("HOG", _t_hog, "8×8 image with 4×4 cells → 4 × 9 = 36 features; a vertical edge puts all energy in the 0° bin.")
if ready("HOG"):
    X, y = load_digits(return_X_y=True); imgs = X.reshape(-1, 8, 8) / 16
    H_ = np.array([hog(im, cell=4) for im in imgs])
    rng = np.random.default_rng(0); idx = rng.permutation(len(X)); tr, te = idx[:200], idx[200:]          # only 200 training images
    res = {}
    for name, F_ in [("raw pixels", X / 16), ("HOG", H_), ("pixels + HOG", np.hstack([X / 16, H_]))]:
        res[name] = LinearSVC(C=1, max_iter=20000).fit(F_[tr], y[tr]).score(F_[te], y[te]); print(f"{name:<14} {F_.shape[1]:>3} features: test accuracy {res[name]:.3f}")
    check("features help", lambda: res["pixels + HOG"] > res["raw pixels"], "Adding HOG to the raw pixels should help a linear model trained on 200 images.")

<details><summary>▶ Solution</summary>

```python
def conv2d(img, kernel):
    kh, kw = kernel.shape
    out = np.zeros((img.shape[0] - kh + 1, img.shape[1] - kw + 1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = np.sum(img[i:i + kh, j:j + kw] * kernel)
    return out
```

```python
def hog(img, cell=4, bins=9):
    gx = np.zeros_like(img); gy = np.zeros_like(img)
    gx[:, 1:-1] = img[:, 2:] - img[:, :-2]
    gy[1:-1, :] = img[2:, :] - img[:-2, :]
    mag = np.hypot(gx, gy); ang = np.rad2deg(np.arctan2(gy, gx)) % 180
    feats = []
    for i in range(0, img.shape[0], cell):
        for j in range(0, img.shape[1], cell):
            m = mag[i:i + cell, j:j + cell].ravel(); a = ang[i:i + cell, j:j + cell].ravel()
            h = np.bincount(np.minimum((a / (180 / bins)).astype(int), bins - 1), weights=m, minlength=bins)
            feats.append(h / (np.linalg.norm(h) + 1e-6))
    return np.concatenate(feats)
```
</details>

## Stretch goals
- Vectorise `conv2d` with `np.lib.stride_tricks.sliding_window_view`, and time both.
- Visualise the HOG of a person-shaped photo as little oriented lines per cell.
- Train a tiny CNN on the same 200 digits ([[CNN]]): does it learn Sobel-like first-layer filters?